# Step 1: Research & Data Source Discovery

## 1.1 Environment Setup and Dependencies

We fetch the data from OpenStreetMap. We use the original OSM ID (osmid) as our primary identifier and calculate the exact center point (latitude and longitude) for each location.

* **Primary Source: OpenStreetMap (OSM)**: Used to extract the spatial location of employment agencies.


## 1.2 Data and Boundary Configuration

The project focuses exclusively on data within the **Berlin, Germany** boundary.

* **Spatial Integrity Plan**: Data will be joined to the **Local Reference System (LOR) boundaries** to derive the mandatory `district_id` and `neighborhood_id` for final database compliance.

In [32]:
import pandas as pd
import geopandas as gpd
import osmnx as ox
import os 
import json

# --- 1.1 CONFIGURATION ---
# Using the specific paths and tags from your previous workflow
PLACE_NAME = "Berlin, Germany"
OSM_TAGS = {"office": "employment_agency"}

# Update LOR_PATH to match the exact filename of the GeoJSON you uploaded
#LOR_PATH = "lor_ortsteile (1).geojson" 
#OUTPUT_PATH = "output/jobcenters_berlin.csv"

print("Libraries loaded.")
print(f"Configuration set for {PLACE_NAME} with OSM tags: {OSM_TAGS}")

# --- 1.2 LIVE DATA EXTRACTION (OSM) ---
print("\nFetching live data from OpenStreetMap (Overpass API)...")
try:
    # Fetch data and ensure the coordinate system is standard WGS84 (EPSG:4326)
    jobcenter_data_raw = ox.features_from_place(PLACE_NAME, OSM_TAGS)
    jobcenter_data_raw = gpd.GeoDataFrame(
        jobcenter_data_raw,
        geometry="geometry",
        crs="EPSG:4326"
    )
    print(f"Success! Retrieved {len(jobcenter_data_raw)} features.")
except Exception as e:
    raise RuntimeError(f"OSM extraction failed: {e}")

# --- 1.3 MANDATORY DATA CLEANING ---
# We explicitly check and report on null values in mandatory columns
print("\n--- Diagnostic Check: Nulls in Critical Columns ---")
null_counts = jobcenter_data_raw[['name', 'geometry']].isnull().sum()
print("Missing values in critical columns:")
print(null_counts)

# Drop rows missing 'name' or 'geometry' to enforce database NOT NULL compliance
initial_count = len(jobcenter_data_raw)
jobcenter_enriched = jobcenter_data_raw.dropna(subset=["name", "geometry"]).copy()

dropped_count = initial_count - len(jobcenter_enriched)
print(f"Mandatory Drop: Removed {dropped_count} rows due to missing name/geometry.")

# --- 1.4 COORDINATE PREPARATION ---
# Extract centroids to handle both 'Point' and 'Polygon' features safely
jobcenter_enriched['latitude'] = jobcenter_enriched.geometry.centroid.y
jobcenter_enriched['longitude'] = jobcenter_enriched.geometry.centroid.x

print("\n--- Step 1 Complete ---")
print(jobcenter_enriched[['name', 'latitude', 'longitude']].head())

Libraries loaded.
Configuration set for Berlin, Germany with OSM tags: {'office': 'employment_agency'}

Fetching live data from OpenStreetMap (Overpass API)...
Success! Retrieved 65 features.

--- Diagnostic Check: Nulls in Critical Columns ---
Missing values in critical columns:
name        2
geometry    0
dtype: int64
Mandatory Drop: Removed 2 rows due to missing name/geometry.

--- Step 1 Complete ---
                                               name   latitude  longitude
element id                                                               
node    275368512   Jobcenter Mitte am Leopoldplatz  52.546772  13.356516
        1211913324           Arbeitsagentur Spandau  52.533775  13.186554
        1340158173        Jobcenter Berlin Neukölln  52.478975  13.427887
        1450906609               Agentur für Arbeit  52.578452  13.308718
        2277566662               Agentur für Arbeit  52.456592  13.411478


In [33]:
jobcenter_data_raw.head()

geometry addr:city addr:country  \
element id                                                             
node    275368512   POINT (13.35652 52.54677)    Berlin           DE   
        1211913324  POINT (13.18655 52.53378)    Berlin           DE   
        1340158173  POINT (13.42789 52.47898)    Berlin           DE   
        1450906609  POINT (13.30872 52.57845)    Berlin           DE   
        2277566662  POINT (13.41148 52.45659)       NaN          NaN   

                                     addr:housename addr:housenumber  \
element id                                                             
node    275368512   Jobcenter Mitte am Leopoldplatz              147   
        1211913324                              NaN            75-77   
        1340158173                              NaN               27   
        1450906609                              NaN               40   
        2277566662                              NaN              NaN   

                   addr:postcode         addr:street  addr:suburb  \
element id                                                          
node    275368512          13353        Müllerstraße      Wedding   
        1211913324         13581  Brunsbütteler Damm      Spandau   
        1340158173         12053      Mainzer Straße     Neukölln   
        1450906609         13509       Innungsstraße  Borsigwalde   
        2277566662           NaN                 NaN          NaN   

                                 brand brand:wikidata  ... lda:criteria  \
element id                                             ...                
node    275368512            Jobcenter      Q56292847  ...          NaN   
        1211913324                 NaN            NaN  ...          NaN   
        1340158173           Jobcenter      Q56292847  ...          NaN   
        1450906609  Agentur für Arbeit       Q1478016  ...          NaN   
        2277566662  Agentur für Arbeit       Q1478016  ...          NaN   

                   ref:lda height air_conditioning toilets alt_name name:de  \
element id                                                                    
node    275368512      NaN    NaN              NaN     NaN      NaN     NaN   
        1211913324     NaN    NaN              NaN     NaN      NaN     NaN   
        1340158173     NaN    NaN              NaN     NaN      NaN     NaN   
        1450906609     NaN    NaN              NaN     NaN      NaN     NaN   
        2277566662     NaN    NaN              NaN     NaN      NaN     NaN   

                   type government wikidata  
element id                                   
node    275368512   NaN        NaN      NaN  
        1211913324  NaN        NaN      NaN  
        1340158173  NaN        NaN      NaN  
        1450906609  NaN        NaN      NaN  
        2277566662  NaN        NaN      NaN  

[5 rows x 62 columns]

In [34]:
jobcenter_data_raw = jobcenter_data_raw.reset_index()

In [35]:
jobcenter_data_raw.columns

Index(['element', 'id', 'geometry', 'addr:city', 'addr:country',
       'addr:housename', 'addr:housenumber', 'addr:postcode', 'addr:street',
       'addr:suburb', 'brand', 'brand:wikidata', 'check_date:opening_hours',
       'contact:phone', 'contact:website', 'name', 'office', 'opening_hours',
       'wheelchair', 'toilets:wheelchair', 'website', 'branch', 'check_date',
       'email', 'opening_hours:signed', 'operator', 'phone', 'brand:wikipedia',
       'internet_access', 'internet_access:fee', 'internet_access:ssid',
       'official_name', 'smoking', 'source', 'contact:email', 'contact:fax',
       'description', 'operator:type', 'short_name', 'level', 'note',
       'addr:floor', 'building:levels', 'image', 'building', 'building:colour',
       'roof:levels', 'roof:shape', 'old_name', 'landuse', 'addr:place',
       'heritage', 'heritage:operator', 'heritage:website', 'lda:criteria',
       'ref:lda', 'height', 'air_conditioning', 'toilets', 'alt_name',
       'name:de', 'type',

In [36]:
jobcenter_enriched.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
MultiIndex: 63 entries, ('node', np.int64(275368512)) to ('way', np.int64(1092876804))
Data columns (total 64 columns):
 #   Column                    Non-Null Count  Dtype   
---  ------                    --------------  -----   
 0   geometry                  63 non-null     geometry
 1   addr:city                 50 non-null     object  
 2   addr:country              28 non-null     object  
 3   addr:housename            2 non-null      object  
 4   addr:housenumber          51 non-null     object  
 5   addr:postcode             50 non-null     object  
 6   addr:street               50 non-null     object  
 7   addr:suburb               27 non-null     object  
 8   brand                     27 non-null     object  
 9   brand:wikidata            27 non-null     object  
 10  check_date:opening_hours  12 non-null     object  
 11  contact:phone             4 non-null      object  
 12  contact:website           6 non-null      obj

## Step 2: Cleanup & Removing Redundancy
Explanation: Here we drop city and country because they are redundant for a Berlin project. We also remove contact:website and operator:type to keep the schema lean.
Why drop contact "website"

Maintenance: External URLs like websites change frequently. If you include them in the primary table now, the data becomes "stale" very quickly.

Scope: The current goal is to map the job centers to the Berlin District LOR system. Extra information like websites or phone numbers can be added in a later "enrichment" task once the primary table structure is approved.

Additionally: The operator:type column is a classification tag in OpenStreetMap. It tells the database who runs the facility. In the context of Berlin Job Centers, this usually indicates public. 
The center is a government-run entity (e.g., the Bundesagentur für Arbeit or local municipal government). Most Job Centers fall into this category.

In [37]:
# 1. Identify redundant columns to drop
cols_to_drop = ['addr:city', 'addr:country', 'wikidata', 'operator:type', 'contact:website', 'element']

# FIX: Changed 'gdf_raw' to 'jobcenter_data_raw' to match your notebook
jobcenter_clean = jobcenter_data_raw.drop(columns=[c for c in cols_to_drop if c in jobcenter_data_raw.columns])

# 2. Rename 'name' to 'center_name' for SQL standards and remove empty rows
jobcenter_clean = jobcenter_clean.dropna(subset=['name']).copy()
jobcenter_clean = jobcenter_clean.rename(columns={'name': 'center_name'})

# --- Verify Step 2 ---
print("--- Check Column Names ---")
print(jobcenter_clean.columns.tolist())
print("\n--- Rows remaining after cleaning ---")
print(len(jobcenter_clean))

--- Check Column Names ---
['id', 'geometry', 'addr:housename', 'addr:housenumber', 'addr:postcode', 'addr:street', 'addr:suburb', 'brand', 'brand:wikidata', 'check_date:opening_hours', 'contact:phone', 'center_name', 'office', 'opening_hours', 'wheelchair', 'toilets:wheelchair', 'website', 'branch', 'check_date', 'email', 'opening_hours:signed', 'operator', 'phone', 'brand:wikipedia', 'internet_access', 'internet_access:fee', 'internet_access:ssid', 'official_name', 'smoking', 'source', 'contact:email', 'contact:fax', 'description', 'short_name', 'level', 'note', 'addr:floor', 'building:levels', 'image', 'building', 'building:colour', 'roof:levels', 'roof:shape', 'old_name', 'landuse', 'addr:place', 'heritage', 'heritage:operator', 'heritage:website', 'lda:criteria', 'ref:lda', 'height', 'air_conditioning', 'toilets', 'alt_name', 'name:de', 'type', 'government']

--- Rows remaining after cleaning ---
63


## Step 3: Spatial Mapping (District Join)

Explanation: Load the official Berlin district file and  use a Spatial Join to see which district polygon each job center point "falls into." This gives us the neighborhood and district names automatically.

In [38]:
LOR_PATH = "lor_ortsteile.geojson"
lor_gdf = gpd.read_file(LOR_PATH).to_crs(epsg=4326)

In [39]:
import os
print("LOR file exists:", os.path.exists(LOR_PATH))

LOR file exists: True


In [40]:
import geopandas as gpd

# 1. Rename columns based on the 'lor_ortsteile' properties found in the file
lor_gdf = lor_gdf.rename(columns={
    "BEZIRK": "district",
    "OTEIL": "neighborhood",
    "spatial_name": "neighborhood_id"
})

# 2. Spatial Join: Mapping Job Center points to District polygons
# This uses the cleaned 'jobcenter_clean' data from your previous cell
jobcenter_mapped = gpd.sjoin(
    jobcenter_clean.reset_index(drop=True), 
    lor_gdf[['district', 'neighborhood', 'neighborhood_id', 'geometry']], 
    how='left', 
    predicate='within'
)

# --- Verification ---
print("--- Check Mapping Results ---")
print(jobcenter_mapped['district'].value_counts())

print("\n--- Check Mapped Data Preview ---")
print(jobcenter_mapped[['center_name', 'district', 'neighborhood']].head())

--- Check Mapping Results ---
district
Mitte                         14
Charlottenburg-Wilmersdorf    10
Friedrichshain-Kreuzberg       9
Neukölln                       7
Spandau                        4
Tempelhof-Schöneberg           4
Steglitz-Zehlendorf            4
Pankow                         4
Marzahn-Hellersdorf            3
Treptow-Köpenick               2
Reinickendorf                  1
Lichtenberg                    1
Name: count, dtype: int64

--- Check Mapped Data Preview ---
                       center_name              district neighborhood
0  Jobcenter Mitte am Leopoldplatz                 Mitte      Wedding
1           Arbeitsagentur Spandau               Spandau      Spandau
2        Jobcenter Berlin Neukölln              Neukölln     Neukölln
3               Agentur für Arbeit         Reinickendorf  Borsigwalde
4               Agentur für Arbeit  Tempelhof-Schöneberg    Tempelhof


## 4: Stable ID Generation and District Mapping
Deterministic Stable ID: A persistent, numeric-only ID is generated using hashlib.sha256. By hashing the geographic centroid, we ensure IDs are unique and immutable, avoiding previous AttributeError issues with different geometry types.

Official District Mapping: To comply with the final data pool schema, we map administrative district names to their official 8-digit numeric IDs (e.g., Mitte = 11001001). This ensures the data is ready for SQL relational joins.hment:** The `enrich_data_from_wikidata` function is applied to fill the `operator_name` and `contact_website` columns.

In [41]:
jobcenter_mapped.head()

,id,geometry,addr:housename,addr:housenumber,addr:postcode,addr:street,addr:suburb,brand,brand:wikidata,check_date:opening_hours,...,air_conditioning,toilets,alt_name,name:de,type,government,index_right,district,neighborhood,neighborhood_id
0,275368512,POINT (13.35652 52.54677),Jobcenter Mitte am Leopoldplatz,147,13353,Müllerstraße,Wedding,Jobcenter,Q56292847,2025-04-16,...,NaN,NaN,NaN,NaN,NaN,NaN,4,Mitte,Wedding,0105
1,1211913324,POINT (13.18655 52.53378),NaN,75-77,13581,Brunsbütteler Damm,Spandau,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,28,Spandau,Spandau,0501
2,1340158173,POINT (13.42789 52.47898),NaN,27,12053,Mainzer Straße,Neukölln,Jobcenter,Q56292847,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,50,Neukölln,Neukölln,0801
3,1450906609,POINT (13.30872 52.57845),NaN,40,13509,Innungsstraße,Borsigwalde,Agentur für Arbeit,Q1478016,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,95,Reinickendorf,Borsigwalde,1211
4,2277566662,POINT (13.41148 52.45659),NaN,NaN,NaN,NaN,NaN,Agentur für Arbeit,Q1478016,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,46,Tempelhof-Schöneberg,Tempelhof,0703


In [42]:
import hashlib

# --- 4.1 DEFINITIONS (Write once) ---
def generate_stable_id(name, lat, lon):
    """Generates a unique 10-digit ID based on name and coordinates."""
    input_data = f"{name}_{lat}_{lon}".encode('utf-8')
    hash_hex = hashlib.sha256(input_data).hexdigest()
    return int(hash_hex, 16) % (10**10)

district_mapping = {
    'Mitte': '11001001', 'Friedrichshain-Kreuzberg': '11002002',
    'Pankow': '11003003', 'Charlottenburg-Wilmersdorf': '11004004',
    'Spandau': '11005005', 'Steglitz-Zehlendorf': '11006006',
    'Tempelhof-Schöneberg': '11007007', 'Neukölln': '11008008',
    'Treptow-Köpenick': '11009009', 'Marzahn-Hellersdorf': '11010010',
    'Lichtenberg': '11011011', 'Reinickendorf': '11012012'
}

# --- 4.2 EXECUTION (The Calls) ---
# 1. Coordinate Prep (Ensuring columns exist)
jobcenter_mapped['latitude'] = jobcenter_mapped.geometry.centroid.y
jobcenter_mapped['longitude'] = jobcenter_mapped.geometry.centroid.x

# 2. Call the Stable ID function
print("Generating stable IDs...")
jobcenter_mapped['id'] = jobcenter_mapped.apply(
    lambda row: generate_stable_id(row['center_name'], row['latitude'], row['longitude']), 
    axis=1
)

# 3. Call the District mapping
print("Mapping districts...")
jobcenter_mapped['district_id'] = jobcenter_mapped['district'].map(district_mapping)

print("Step 4 complete. Data is enriched and identified.")
print("Mapping district names to official IDs...")
jobcenter_mapped['district_id'] = jobcenter_mapped['district'].map(district_mapping).astype(str)

# --- Verification ---
print("\n--- Step 4 Verification ---")
print(jobcenter_mapped[['id', 'center_name', 'district', 'district_id']].head())

Generating stable IDs...
Mapping districts...
Step 4 complete. Data is enriched and identified.
Mapping district names to official IDs...

--- Step 4 Verification ---
           id                      center_name              district  \
0  6660665090  Jobcenter Mitte am Leopoldplatz                 Mitte   
1  1092468394           Arbeitsagentur Spandau               Spandau   
2   730832232        Jobcenter Berlin Neukölln              Neukölln   
3   246338546               Agentur für Arbeit         Reinickendorf   
4  6239357044               Agentur für Arbeit  Tempelhof-Schöneberg   

  district_id  
0    11001001  
1    11005005  
2    11008008  
3    11012012  
4    11007007  


In [43]:
jobcenter_mapped.head()

,id,geometry,addr:housename,addr:housenumber,addr:postcode,addr:street,addr:suburb,brand,brand:wikidata,check_date:opening_hours,...,name:de,type,government,index_right,district,neighborhood,neighborhood_id,latitude,longitude,district_id
0,6660665090,POINT (13.35652 52.54677),Jobcenter Mitte am Leopoldplatz,147,13353,Müllerstraße,Wedding,Jobcenter,Q56292847,2025-04-16,...,NaN,NaN,NaN,4,Mitte,Wedding,0105,52.546772,13.356516,11001001
1,1092468394,POINT (13.18655 52.53378),NaN,75-77,13581,Brunsbütteler Damm,Spandau,NaN,NaN,NaN,...,NaN,NaN,NaN,28,Spandau,Spandau,0501,52.533775,13.186554,11005005
2,730832232,POINT (13.42789 52.47898),NaN,27,12053,Mainzer Straße,Neukölln,Jobcenter,Q56292847,NaN,...,NaN,NaN,NaN,50,Neukölln,Neukölln,0801,52.478975,13.427887,11008008
3,246338546,POINT (13.30872 52.57845),NaN,40,13509,Innungsstraße,Borsigwalde,Agentur für Arbeit,Q1478016,NaN,...,NaN,NaN,NaN,95,Reinickendorf,Borsigwalde,1211,52.578452,13.308718,11012012
4,6239357044,POINT (13.41148 52.45659),NaN,NaN,NaN,NaN,NaN,Agentur für Arbeit,Q1478016,NaN,...,NaN,NaN,NaN,46,Tempelhof-Schöneberg,Tempelhof,0703,52.456592,13.411478,11007007


## 5: Data Standardization and Final Export
Schema Compliance: The final dataset is filtered to include only the mandatory 8 columns required for the database pool: id, district_id, center_name, latitude, longitude, neighborhood, district, and neighborhood_id.

WKT & Coordinate Prep: Coordinates are extracted from the geometric centroids and formatted as numeric floats, ensuring compatibility with standard SQL spatial types.

Stable ID Integration: The deterministic IDs generated in Step 4 are finalized as the primary keys for this dataset.

Data Source Attribution: A data_source tag (OSM_LOR) is appended to ensure traceability for future audits.

In [44]:
import os

# --- 5.1 SPATIAL DATA PREP ---
# Safety check: Remove any rows with missing shapes to prevent WKT errors
jobcenter_mapped = jobcenter_mapped.dropna(subset=['geometry']).copy()

# Generate the geometry column in WKT format (e.g., POINT (13.4 52.5))
# We name it 'geometry' directly as requested
jobcenter_mapped['geometry'] = jobcenter_mapped['geometry'].apply(
    lambda x: x.wkt if x is not None else None
)

# --- 5.2 FINAL SCHEMA SELECTION ---
final_columns = [
    'id', 
    'district_id', 
    'center_name', 
    'latitude', 
    'longitude', 
    'geometry',  # Clean column name for SQL
    'neighborhood', 
    'district', 
    'neighborhood_id'
]

# Create the final dataframe and add the source tag
df_final = jobcenter_mapped[final_columns].copy()
df_final['data_source'] = 'OSM_LOR'

# --- 5.3 VERIFICATION & EXPORT ---
print("--- Final Data Audit ---")
print(f"Total Records: {len(df_final)}")
print(f"Columns to Export: {df_final.columns.tolist()}")

# Preview the head to make sure 'geometry' is there
print("\n--- Data Preview ---")
print(df_final[['center_name', 'geometry']].head())

# Export to CSV
os.makedirs("output", exist_ok=True)
output_path = "output/jobcenters_berlin_final.csv"
df_final.to_csv(output_path, index=False)

print(f"\n✅ SUCCESS: Final file with column 'geometry' saved to {output_path}")

--- Final Data Audit ---
Total Records: 63
Columns to Export: ['id', 'district_id', 'center_name', 'latitude', 'longitude', 'geometry', 'neighborhood', 'district', 'neighborhood_id', 'data_source']

--- Data Preview ---
                       center_name                       geometry
0  Jobcenter Mitte am Leopoldplatz  POINT (13.3565162 52.5467722)
1           Arbeitsagentur Spandau  POINT (13.1865537 52.5337752)
2        Jobcenter Berlin Neukölln  POINT (13.4278868 52.4789752)
3               Agentur für Arbeit  POINT (13.3087179 52.5784523)
4               Agentur für Arbeit  POINT (13.4114781 52.4565923)

✅ SUCCESS: Final file with column 'geometry' saved to output/jobcenters_berlin_final.csv


In [45]:
%pip install sqlalchemy psycopg2-binary

Note: you may need to restart the kernel to use updated packages.


In [46]:
print(jobcenter_mapped.columns.tolist())

['id', 'geometry', 'addr:housename', 'addr:housenumber', 'addr:postcode', 'addr:street', 'addr:suburb', 'brand', 'brand:wikidata', 'check_date:opening_hours', 'contact:phone', 'center_name', 'office', 'opening_hours', 'wheelchair', 'toilets:wheelchair', 'website', 'branch', 'check_date', 'email', 'opening_hours:signed', 'operator', 'phone', 'brand:wikipedia', 'internet_access', 'internet_access:fee', 'internet_access:ssid', 'official_name', 'smoking', 'source', 'contact:email', 'contact:fax', 'description', 'short_name', 'level', 'note', 'addr:floor', 'building:levels', 'image', 'building', 'building:colour', 'roof:levels', 'roof:shape', 'old_name', 'landuse', 'addr:place', 'heritage', 'heritage:operator', 'heritage:website', 'lda:criteria', 'ref:lda', 'height', 'air_conditioning', 'toilets', 'alt_name', 'name:de', 'type', 'government', 'index_right', 'district', 'neighborhood', 'neighborhood_id', 'latitude', 'longitude', 'district_id']


In [47]:
import psycopg2
from sqlalchemy import create_engine, text
import warnings

warnings.filterwarnings("ignore")

In [48]:
user_name=''
password=''

In [49]:
# 1. Configuration (using the stable 127.0.0.1 address)
user_name = 'tigist_hayilemariyam'
password = 'tc3WUbE1DZ6SzYZ'
host = '127.0.0.1' 
port = '5433'
database = 'layereddb'
schema = 'berlin_source_data'
table_name = 'job_centers'

In [50]:
engine = create_engine(f'postgresql+psycopg2://{user_name}:{password}@{host}:{port}/{database}')

In [58]:
import pandas as pd
from sqlalchemy import create_engine, text

# 1. Define the SQL Blueprint
# We name it here so Python knows what 'create_table_query' is
create_table_query = """
DROP TABLE IF EXISTS berlin_source_data.job_centers CASCADE;

CREATE TABLE berlin_source_data.job_centers (
    id TEXT PRIMARY KEY,
    district_id TEXT,
    center_name TEXT,
    latitude DOUBLE PRECISION,
    longitude DOUBLE PRECISION,
    geometry TEXT,
    neighborhood TEXT,
    district TEXT,
    neighborhood_id TEXT,
    data_source TEXT,
    CONSTRAINT fk_district FOREIGN KEY (district_id) 
        REFERENCES berlin_source_data.districts (district_id)
);
"""

# 2. Execute the Query
try:
    with engine.connect() as conn:
        # This line will now work because we defined it above!
        conn.execute(text(create_table_query))
        conn.commit()
        print(" SUCCESS: The table has been created in the database!")
except NameError:
    print("Error: The 'engine' variable isn't defined. Run your connection cell first!")
except Exception as e:
    print(f" Database Error: {e}")

 SUCCESS: The table has been created in the database!


In [59]:
# Prepare the final dataframe columns to match the table exactly
df_to_upload = jobcenter_mapped[[
    'id', 'district_id', 'center_name', 'latitude', 'longitude', 
    'geometry', 'neighborhood', 'district', 'neighborhood_id'
]].copy()

# Add the data source tag your boss likes
df_to_upload['data_source'] = 'OSM_LOR'

try:
    # Use the 'append' method since the table already exists
    df_to_upload.to_sql(
        name='job_centers', 
        con=engine, 
        schema='berlin_source_data', 
        if_exists='append', 
        index=False
    )
    print(f" SUCCESS! {len(df_to_upload)} rows uploaded to AWS.")
except Exception as e:
    print(f"❌ Upload Error: {e}")

 SUCCESS! 63 rows uploaded to AWS.


In [60]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM berlin_source_data.job_centers"))
    print(f"Total rows currently in database: {result.scalar()}")

Total rows currently in database: 63


In [62]:
import pandas as pd
from sqlalchemy import text

# This query checks the count and shows a few actual rows 
# to ensure the data didn't get corrupted during the upload.
check_query = """
SELECT 
    district_id, 
    center_name, 
    latitude, 
    longitude, 
    data_source
FROM berlin_source_data.job_centers
ORDER BY district_id ASC
LIMIT 10;
"""

try:
    with engine.connect() as conn:
        df_audit = pd.read_sql(text(check_query), conn)
        
        print(f"Connection Verified.")
        print(f" Displaying a sample of the 63 rows currently in AWS:")
        display(df_audit)
        
except Exception as e:
    print(f" Could not reach the database: {e}")

Connection Verified.
 Displaying a sample of the 63 rows currently in AWS:


,district_id,center_name,latitude,longitude,data_source
0,11001001,Players Agentur Management,52.525917,13.400791,OSM_LOR
1,11001001,Kraftfahrer-Agentur,52.555671,13.346877,OSM_LOR
2,11001001,Job-Point,52.525535,13.339522,OSM_LOR
3,11001001,Agentur Neidig,52.530159,13.340631,OSM_LOR
4,11001001,Beta gGmbH,52.549239,13.364556,OSM_LOR
5,11001001,Zenjob,52.527579,13.343648,OSM_LOR
6,11001001,Jobcenter Mitte am Leopoldplatz,52.546772,13.356516,OSM_LOR
7,11001001,Union Sozialer Einrichtungen,52.558149,13.378352,OSM_LOR
8,11001001,Jobcenter,52.510523,13.402383,OSM_LOR
9,11001001,Agentur Schlag,52.514923,13.337123,OSM_LOR


In [63]:
# This query asks for every single row
full_check_query = "SELECT * FROM berlin_source_data.job_centers ORDER BY district_id;"

with engine.connect() as conn:
    df_all = pd.read_sql(text(full_check_query), conn)

# This tells the notebook to show all 63 rows without hiding any
with pd.option_context('display.max_rows', None):
    display(df_all)

,id,district_id,center_name,latitude,longitude,geometry,neighborhood,district,neighborhood_id,data_source
0,2782312819,11001001,Job-Point,52.525535,13.339522,POINT (13.3395222 52.5255348),Moabit,Mitte,0102,OSM_LOR
1,4971452348,11001001,Kraftfahrer-Agentur,52.555671,13.346877,POINT (13.3468772 52.5556712),Wedding,Mitte,0105,OSM_LOR
2,1346016268,11001001,Zenjob,52.527579,13.343648,POINT (13.3436479 52.5275788),Moabit,Mitte,0102,OSM_LOR
3,6660665090,11001001,Jobcenter Mitte am Leopoldplatz,52.546772,13.356516,POINT (13.3565162 52.5467722),Wedding,Mitte,0105,OSM_LOR
4,3137117730,11001001,BSM Personalmanagement,52.535615,13.357524,POINT (13.3575242 52.5356149),Moabit,Mitte,0102,OSM_LOR
5,537096787,11001001,Beta gGmbH,52.549239,13.364556,POINT (13.3645563 52.5492387),Wedding,Mitte,0105,OSM_LOR
6,3608024519,11001001,recrew,52.520063,13.393426,POINT (13.3934256 52.5200629),Mitte,Mitte,0101,OSM_LOR
7,5424095615,11001001,Jobcenter Berlin Mitte,52.543833,13.365058,"POLYGON ((13.3653305 52.5435849, 13.3657019 52...",Wedding,Mitte,0105,OSM_LOR
8,7211656755,11001001,Agentur Schlag,52.514923,13.337123,POINT (13.3371231 52.5149233),Hansaviertel,Mitte,0103,OSM_LOR
9,9462705127,11001001,Players Agentur Management,52.525917,13.400791,POINT (13.4007908 52.525917),Mitte,Mitte,0101,OSM_LOR
